# Modelling the Lunar Magma Ocean (Johnson et al. 2021)

This notebook reproduces the findings from the paper:  
**"The phases of the Moon: Modelling crystallisation of the lunar magma ocean through equilibrium thermodynamics"**  
by T.E. Johnson, L.J. Morrissey, A.A. Nemchin, N.J. Gardiner, and J.F. Snape (*Earth and Planetary Science Letters*, 2021).

### Abstract Overview
The primary internal structure of the Moon reflects crystallisation of a lunar magma ocean (LMO). Johnson et al. (2021) used thermodynamic models in the $K_2O–Na_2O–CaO–FeO–MgO–Al_2O_3–SiO_2–TiO_2–Cr_2O_3$ system to model this process based on two end-member bulk compositions: **Taylor Whole-Moon (TWM)** and **Lunar Primitive Upper Mantle (LPUM)**.

The model follows a two-step approach:
1. **Equilibrium Crystallisation (Stage 0):** The first 50 vol.% of the LMO crystallises while crystals remain suspended in the convecting magma.
2. **Fractional Crystallisation (Stages 1-10):** The remaining 50 vol.% melt crystallises in 5 vol.% increments, with crystals either sinking to the mantle or floating to form the crust.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phasetools.models.magma_ocean import MagmaOcean
import sys

## 1. Defining Bulk Compositions

Johnson et al. (2021) model two bulk compositions from Table 1:
- **TWM:** Enriched in refractory lithophile elements (Al, Ca, Ti) and FeO.
- **LPUM:** More Earth-like, depleted in refractory oxides.

The oxides used are: $SiO_2, TiO_2, Al_2O_3, Cr_2O_3, FeO, MgO, CaO, Na_2O, K_2O$.

In [ ]:
oxides = ["SiO2", "TiO2", "Al2O3", "Cr2O3", "FeO", "MgO", "CaO", "Na2O", "K2O"]

# Taylor Whole Moon (TWM)
twm_wt = [44.40, 0.31, 6.14, 0.40, 10.90, 32.70, 4.73, 0.04, 0.01]

# Lunar Primitive Upper Mantle (LPUM)
lpum_wt = [46.10, 0.17, 3.93, 0.38, 7.62, 38.30, 3.18, 0.05, 0.01]

## 2. Stage 0: Equilibrium Crystallisation

> "During the initial stage of equilibrium crystallisation (Stage 0), crystals remain suspended within a vigorously convecting magma until the solid (crystal) fraction reaches 50 vol.%."

We model this from the core-mantle boundary (~45 kbar) to the top of the Stage 0 cumulates (~17.3 kbar) across several pressure intervals.

In [ ]:
def run_stage_0_analysis(name, wt_comp):
    print(f"Running Stage 0 for {name}...")
    # Moon parameters: R=1737.1 km, Core R=330 km, g=1.62 m/s^2, rho_avg=3350 kg/m^3
    mo = MagmaOcean(radius_km=1737.1, core_radius_km=330.0, gravity=1.62, avg_density=3350.0, verbose=False)
    mo.setup_bulk_composition(oxides, wt_comp, sys_in="wt")
    
    # Stage 0: 45 kbar to 17.3 kbar
    results_0, avg_melt = mo.run_stage_0(p_start=45.0, p_end=17.3, solid_frac=0.5, p_intervals=10)
    return results_0, avg_melt, mo

results_twm, melt_twm, mo_twm = run_stage_0_analysis("TWM", twm_wt)
results_lpum, melt_lpum, mo_lpum = run_stage_0_analysis("LPUM", lpum_wt)

### Plotting Stage 0 Mineralogy

We can now compare the mineral assemblages. A key finding of the paper is that **TWM contains significant garnet in the deep mantle**, whereas **LPUM has none**.

In [ ]:
def plot_stage_0(results, title):
    all_phases = sorted(list(set().union(*[m.keys() for m in results["modes"]])))
    p = results["P"]
    modes_data = [[m.get(ph, 0) for m in results["modes"]] for ph in all_phases]
    
    plt.figure(figsize=(10, 6))
    plt.stackplot(p, modes_data, labels=all_phases, alpha=0.8)
    plt.xlabel("Pressure (kbar)")
    plt.ylabel("Volume Fraction")
    plt.title(title)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.gca().invert_xaxis()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_stage_0(results_twm, "Stage 0: Taylor Whole Moon (TWM)")
plot_stage_0(results_lpum, "Stage 0: Lunar Primitive Upper Mantle (LPUM)")

## 3. Stages 1-10: Fractional Crystallisation

> "We then simulate fractional crystallisation of the remaining 50 vol.% melt in ten stages (stages 1–10), in each case crystallising an identical volume of melt (i.e. 5 vol.% of the starting volume of the LMO)."

Minerals like olivine and pyroxene sink, while plagioclase (feldspar) floats to form the crust.

### Implementation Note: Volume Scaling
To replicate the results of Johnson et al. (2021) exactly, we must ensure that the volume crystallized in each stage (`vol_step`) is scaled relative to the **total volume of the Lunar Magma Ocean**, not just the remaining melt. 

If Stage 0 ends with 50 vol.% crystals, then the remaining melt ocean is 50% of the total. A `vol_step` of 0.05 (5%) of the *total* LMO represents 10% of the *remaining* melt ocean. By using `starting_vol_frac=0.5` in our model, we ensure that after 10 stages of 5% each, the melt ocean is 100% crystallized, representing the **final residual melt (urKREEP)**.

In [ ]:
print("Running Fractional Stages for TWM...")
results_frac_twm = mo_twm.run_fractional_stages(melt_twm, p_start=17.3, p_end=0.01, vol_step=0.05, starting_vol_frac=0.5, n_stages=10)

for res in results_frac_twm:
    print(f"Stage {res['stage']}: P_base {res['p_base']:.1f} kbar (P_top {res['p_top']:.1f} kbar). "
          f"Major phases: {', '.join([f'{ph} ({val*100:.1f}%)' for ph, val in res['layer_modes'].items() if val > 0.01])}")

## 4. Synthesis: The Origin of KREEP

The **final residual melt (urKREEP)** is the highly evolved liquid that becomes trapped between the upward-growing mantle cumulates and the downward-growing anorthositic crust. 

### Geometry Evolution
Following Johnson et al. (2021), we model each stage by calculating the mineral equilibrium of the **whole liquid ocean** at its base pressure. As minerals crystallise:
1. **Sinking phases** (olivine, pyroxene) increase the radius of the solid mantle cumulate pile, raising the base pressure of the ocean.
2. **Floating phases** (plagioclase) decrease the radius of the liquid ocean from the top down, thickening the anorthosite crust.

In our model, Stage 1 starts at **17.3 kbar** (the top of the Stage 0 cumulates). By Stage 7, the base of the liquid ocean has shallowed to **~8 kbar** (0.8 GPa), matching the progression shown in Figure 4 of the paper.

Let's identify when the flotation crust begins to form in the TWM model.

In [ ]:
pl_stage = next((res['stage'] for res in results_frac_twm if 'pl' in res['layer_modes'] or 'fsp' in res['layer_modes']), None)
if pl_stage:
    print(f"Plagioclase flotation crust began forming at Stage {pl_stage} (~{50 + 5*pl_stage}% PCS).")
else:
    print("Plagioclase flotation crust did not form in these stages.")

## Conclusion

Using `phasetools` and the `MagmaOcean` model, we have successfully replicated the first-order features of the lunar interior as proposed by Johnson et al. (2021):
1. **TWM vs LPUM:** The fertile TWM composition stabilises garnet in the deep mantle, while the depleted LPUM does not.
2. **Crystallisation Sequence:** We tracked the evolution from an olivine-dominated lower mantle to an upper mantle containing pyroxenes and eventually plagioclase.
3. **Density Structure:** The model accounts for the geometric changes as layers of different mineralogy and density are added to the cumulate pile.